In [2]:
import numpy as np
import pandas as pd
from scipy import stats
import random
from datetime import datetime, timedelta
from faker import Faker


In [3]:
print(random.seed(42))

None


In [4]:

# ── Reproductibilité ─────────────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)
fake = Faker()
Faker.seed(42)

# ── Paramètres ──────────────────────────────────────────────────────────────
NB_USERS = 500
INTERESTS = ["fitness", "musique", "technologie", "cuisine", "voyage", "cinema"]
INTEREST_PROBS = [0.25, 0.22, 0.18, 0.15, 0.12, 0.08]

ACTIONS = {
    "fitness":     ["watched workout video", "bought protein powder",
                    "liked gym post", "joined gym"],
    "musique":     ["watched music video", "bought headphones",
                    "liked song", "attended concert", "streamed album"],
    "technologie": ["watched AI talk", "read tech blog", "bought laptop",
                    "bought smartwatch", "liked AI article",
                    "contributed to open source"],
    "cuisine":     ["watched cooking show", "bought ingredients",
                    "liked recipe post", "tried new recipe",
                    "bought kitchen gadget"],
    "voyage":      ["watched travel vlog", "booked flight", "bought luggage",
                    "liked travel photo", "checked into hotel"],
    "cinema":      ["watched movie", "watched trailer", "bought streaming sub",
                    "rated film", "bought cinema ticket"],
}

# ── Inférence du type d'action ───────────────────────────────────────────────
def infer_action_type(action):
    """Déduit le type d'action à partir du verbe."""
    if isinstance(action, str):
        if action.startswith(("watched", "read", "streamed")):
            return "view"
        if action.startswith("bought") or action == "booked flight":
            return "buy"
        if action.startswith("liked") or action == "rated film":
            return "like"
    return "other"

# ── Génération des profils ────────────────────────────────
# ── Génération des profils ────────────────────────────────
def generate_users(n=NB_USERS):
    profiles = []
    for i in range(n):
        age = int(np.clip(np.random.normal(loc=35, scale=10), 18, 65))
        nb_interests = random.randint(1, 6)
        interests = list(np.random.choice(
            INTERESTS, size=nb_interests, replace=False, p=INTEREST_PROBS
        ))
        # Générer l'activity_log directement dans le profil
        activity_log = []
        for _ in range(random.randint(5, 15)):
            # 80% du temps : activité dans ses intérêts
            # 20% du temps : activité hors intérêts (comme dans la réalité)
            if random.random() < 0.80:
                category = random.choice(interests)
            else:
                other = [c for c in INTERESTS if c not in interests]
                if other:
                    category = random.choice(other)
                else:
                    category = random.choice(interests)
            activity_log.append(random.choice(ACTIONS[category]))

        profiles.append({
            "name":         fake.name(),
            "age":          age,
            "interests":    interests,
            "activity_log": activity_log,
        })
    return profiles

# ── Conversion en DataFrames pour l'analyse ─────────────────────────────────
def profiles_to_dataframes(profiles):
    """Convertit les profils en 2 DataFrames : users + logs."""
    user_records = []
    log_records  = []
    base_date = datetime(2024, 1, 1)

    for idx, profile in enumerate(profiles):
        # DataFrame utilisateurs
        user_records.append({
            "user_id":   idx + 1,
            "name":      profile["name"],
            "age":       profile["age"],
            "interests": profile["interests"],
        })

        # DataFrame logs
        for action in profile["activity_log"]:
            action_type = infer_action_type(action)
            # Déduire la catégorie de l'action
            category = None
            for cat, actions in ACTIONS.items():
                if action in actions:
                    category = cat
                    break

            delta = timedelta(days=random.randint(0, 180),
                              hours=random.randint(0, 23))
            ts = base_date + delta
            log_records.append({
                "user_id":     idx + 1,
                "action":      action,
                "action_type": action_type,
                "category":    category,
                "timestamp":   ts,
                "hour":        ts.hour,
            })

    users_df = pd.DataFrame(user_records)
    logs_df  = pd.DataFrame(log_records)
    return users_df, logs_df

# ── Injection de bruit (doublons + NaN) ──────────────────────────────────────
def inject_noise(df, dup_fraction=0.03, nan_fraction=0.02):
    """Injecte des doublons et des NaN pour simuler des données réelles."""
    df_noisy = df.copy()

    # ── Doublons ──
    n_dup = int(len(df_noisy) * dup_fraction)
    dup_rows = df_noisy.sample(n=n_dup, random_state=42)
    df_noisy = pd.concat([df_noisy, dup_rows], ignore_index=True)

    # ── Valeurs manquantes ──
    n_nan = int(len(df_noisy) * nan_fraction)
    nan_indices = df_noisy.sample(n=n_nan, random_state=42).index
    for idx in nan_indices:
        col = random.choice(["action", "category", "hour"])
        df_noisy.loc[idx, col] = np.nan

    return df_noisy

# ── Exécution ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # 1. Générer les profils (format dict comme le sujet)
    profiles = generate_users()

    # 2. Afficher un profil exemple
    print("=== Exemple de profil ===")
    print(profiles[0])

    # 3. Convertir en DataFrames pour l'analyse
    users_df, logs_df = profiles_to_dataframes(profiles)
    logs_df = inject_noise(logs_df)

    print("\n=== DataFrame Utilisateurs ===")
    print(users_df)
    print(f"\nShape : {users_df.shape}")

    print("\n=== DataFrame Logs ===")
    print(logs_df.head())
    print(f"\nShape : {logs_df.shape}")
    print(f"Valeurs manquantes :\n{logs_df.isnull().sum()}")
    print(f"Doublons : {logs_df.duplicated().sum()}")

=== Exemple de profil ===
{'name': 'Allison Hill', 'age': 39, 'interests': [np.str_('cuisine'), np.str_('technologie'), np.str_('fitness'), np.str_('voyage'), np.str_('musique'), np.str_('cinema')], 'activity_log': ['bought protein powder', 'watched movie', 'watched music video', 'watched cooking show', 'liked AI article', 'bought headphones']}

=== DataFrame Utilisateurs ===
     user_id                 name  age  \
0          1         Allison Hill   39   
1          2          Noah Rhodes   33   
2          3      Angie Henderson   24   
3          4        Daniel Wagner   38   
4          5      Cristian Santos   28   
..       ...                  ...  ...   
495      496  Lawrence Harrington   39   
496      497       Austin Osborne   34   
497      498        James Bradley   41   
498      499          Meghan Rush   19   
499      500        Phillip Berry   31   

                                             interests  
0    [cuisine, technologie, fitness, voyage, musiqu...  
1 